In [ ]:
from datadings.reader import MsgpackReader
reader =  MsgpackReader('./data/img-dds-val/val.msgpack')

In [ ]:
from datadings.torch import CompressedToPIL
transform = CompressedToPIL()
s = reader[0]
print(s["label"])
img = transform(s["image"])

In [ ]:
from collections import defaultdict
data_dict = defaultdict(list)
for i in range(len(reader)):
    sample = reader[i]
    data_dict[sample["label"]].append((i, sample["key"]))

In [ ]:
len(data_dict.keys())

In [ ]:
keylist = list(data_dict.keys())

In [ ]:
import random
random.sample(data_dict[722], 1)

In [ ]:
img

In [ ]:
len(reader)

In [ ]:
from datadings.torch import CompressedToPIL
transform = CompressedToPIL()
for sample in reader:
    print(sample["label"])
    img = transform(sample["image"])
    break

In [ ]:
import torch
from torch.utils.data import Dataset
import webdataset as wds
import random
import numpy as np
import itertools


class MemoryEfficientImageNetDataset(Dataset):
    def __init__(self, dataset_path, num_classes=10, images_per_class=10):
        """
        Memory-efficient ImageNet WebDataset using standard Dataset
        
        Args:
            dataset_path (str): Path to WebDataset shards
            num_classes (int): Number of random classes to sample
            images_per_class (int): Number of images to sample per class
        """
        self.dataset = wds.WebDataset(dataset_path)
        self.num_classes = num_classes
        self.images_per_class = images_per_class

        # Lightweight class index
        self.class_index = self._build_class_index()

    def _build_class_index(self):
        """
        Build a lightweight index of class positions
        
        Returns:
            dict: Mapping of class labels to sample indices
        """
        class_index = {}
        for idx, sample in enumerate(self.dataset):
            label = sample['cls']
            if label not in class_index:
                class_index[label] = []
            class_index[label].append(idx)
        return class_index

    def __len__(self):
        """Total number of samples per iteration"""
        return self.num_classes * self.images_per_class + 1

    def __getitem__(self, idx):
        """
        Lazy load samples on-demand
        
        Returns:
            tuple: (image tensor, label)
        """
        # Select 10 random classes (only once)
        if not hasattr(self, '_selected_classes'):
            self._selected_classes = random.sample(
                list(self.class_index.keys()), self.num_classes)

        # If index is at the end, select an additional random image
        if idx == self.num_classes * self.images_per_class:
            extra_class = random.choice(self._selected_classes)
            extra_idx = random.choice(self.class_index[extra_class])
            extra_sample = list(itertools.islice(
                self.dataset, extra_idx, extra_idx+1))[0]
            return extra_sample['image'], extra_sample['cls']

        # Sample regular images
        class_idx = idx // self.images_per_class
        image_idx = idx % self.images_per_class

        current_class = self._selected_classes[class_idx]
        sample_global_idx = self.class_index[current_class][image_idx]

        # Lazy load the specific sample
        sample = list(itertools.islice(
            self.dataset, sample_global_idx, sample_global_idx+1))[0]
        return sample['image'], sample['cls']

In [2]:
from utils.cluster_dataloader import ImageNetDataDingsSet2

ds = ImageNetDataDingsSet2(
    data_path="./data/img-dds-val/val.msgpack", num_classes=5, num_samples=200, num_images=10)

/Users/lukasschiesser/Desktop/papers/embed-then-classify/venv/lib/python3.12/site-packages/datadings/reader/msgpack.py:43: UserWarning: data/img-dds-val/val.msgpack.keys not found, some functionality may not be available
  warnings.warn(f'{path} not found, some functionality may not be available')
/Users/lukasschiesser/Desktop/papers/embed-then-classify/venv/lib/python3.12/site-packages/datadings/reader/msgpack.py:43: UserWarning: data/img-dds-val/val.msgpack.key_hashes not found, some functionality may not be available
  warnings.warn(f'{path} not found, some functionality may not be available')
/Users/lukasschiesser/Desktop/papers/embed-then-classify/venv/lib/python3.12/site-packages/datadings/reader/msgpack.py:43: UserWarning: data/img-dds-val/val.msgpack.filter not found, some functionality may not be available
  warnings.warn(f'{path} not found, some functionality may not be available')


In [3]:
import torch
loader = torch.utils.data.DataLoader(
    dataset=ds,
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    shuffle=False
)

In [4]:
for images, labels, pred_image, pred_label in loader:
    print(images.shape)
    print(labels.shape, labels)
    print(pred_image.shape)
    print(pred_label.shape, pred_label)
    break

/Users/lukasschiesser/Desktop/papers/embed-then-classify/venv/lib/python3.12/site-packages/datadings/reader/msgpack.py:43: UserWarning: data/img-dds-val/val.msgpack.keys not found, some functionality may not be available
  warnings.warn(f'{path} not found, some functionality may not be available')
/Users/lukasschiesser/Desktop/papers/embed-then-classify/venv/lib/python3.12/site-packages/datadings/reader/msgpack.py:43: UserWarning: data/img-dds-val/val.msgpack.key_hashes not found, some functionality may not be available
  warnings.warn(f'{path} not found, some functionality may not be available')
/Users/lukasschiesser/Desktop/papers/embed-then-classify/venv/lib/python3.12/site-packages/datadings/reader/msgpack.py:43: UserWarning: data/img-dds-val/val.msgpack.filter not found, some functionality may not be available
  warnings.warn(f'{path} not found, some functionality may not be available')
/Users/lukasschiesser/Desktop/papers/embed-then-classify/venv/lib/python3.12/site-packages/data

torch.Size([16, 50, 3, 224, 224])
torch.Size([16, 50]) tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2,
         2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4,
         4, 4],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2,
         2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4,
         4, 4],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2,
         2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4,
         4, 4],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2,
         2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4,
         4, 4],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2,
         2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4,
         4, 4],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1